#1. Chess Game Engine
Build a simple chess logic system:

Board class with 8x8 grid.

Piece abstract class with is_valid_move method.

Subclasses like Pawn, Rook, Knight with move logic.

Player objects control moves and switch turns.

In [1]:
#Piece Class

from abc import ABC, abstractmethod

class Piece(ABC):
  def __init__(self, color, symbol):
    self.color=color
    self.symbol=symbol
    self.position=None      #pieces are not yet placed on board

  @abstractmethod
  def is_valid_move(self, board, start_pos, end_pos):
    pass

In [2]:
#pawn sub class of piece
class Pawn(Piece):
    def __init__(self, color, symbol):
        super().__init__(color, symbol)
        self.moved = False          #no move is made yet

    def is_valid_move(self, board, start_pos, end_pos):
        start_row, start_col = start_pos
        end_row, end_col = end_pos
        delta_row = end_row - start_row
        delta_col = end_col - start_col

        if board.check_position(end_pos) == False:     #end position should be on board
            print(f'{end_pos} is not on board')
            return False
        if start_pos == end_pos:       #for movement, start and end should be different
            print(f'{start_pos} and {end_pos} are same')
            return False
        if board.check_position(end_pos) != None and board.check_position(end_pos) != False and board.check_position(end_pos).color == self.color:     #end position should be of different color
            print(f'same color is present at {end_pos}')
            return False

        if self.color == 'white':
          #non-capturing
            if delta_row == -1 and delta_col == 0:
                if board.check_position(end_pos) == None:
                    print(f'{end_pos} is empty and can be successfully taken')
                    return True
            elif self.moved == False and delta_row == -2 and delta_col == 0:
                intermediate_row = start_row - 1
                if board.check_position(end_pos)== None and board.check_position((intermediate_row, start_col))== None:
                    print(f'{end_pos} is empty and there is nothing in between, so 2 steps can be successfully taken')
                    return True
            #capturing
            elif delta_row == -1 and (delta_col == -1 or delta_col == 1):
                if board.check_position(end_pos) != None and board.check_position(end_pos) != False: # to capture there must be something
                    print(f'{end_pos} is taken by opposite color and can be captuerd successfully')
                    return True
            print('no condition is satisfied, invalid move')
            return False

        elif self.color == 'black':
            #non-capturing
            if delta_row == 1 and delta_col == 0:
                if board.check_position(end_pos) == None:
                    print(f'{end_pos} is empty and can be successfully taken')
                    return True
            elif self.moved == False and delta_row == 2 and delta_col == 0:
                intermediate_row = start_row + 1
                if board.check_position(end_pos) == None and board.check_position((intermediate_row, start_col)) == None:
                    print(f'{end_pos} is empty and there is nothing in between, so 2 steps can be successfully taken')
                    return True
            #capturing
            elif delta_row == 1 and (delta_col == -1 or delta_col == 1):
                if board.check_position(end_pos) != None and board.check_position(end_pos) != False:
                    print(f'{end_pos} is taken by opposite color and can be captuerd successfully')
                    return True
            print('no condition is satisfied, invalid move')
            return False
        print('invalid color selection')
        return False

In [3]:
#rook sub class of piece
class Rook(Piece):
    def __init__(self, color, symbol):
        super().__init__(color, symbol)
        self.moved = False

    def is_valid_move(self, board, start_pos, end_pos):
        start_row, start_col = start_pos
        end_row, end_col = end_pos
        delta_row = end_row - start_row
        delta_col = end_col - start_col

        if board.check_position(end_pos) == False:      #end position should be on board
            print(f'{end_pos} is not on board')
            return False
        if start_pos == end_pos:      #end position should be different than the starting position
            print(f'{start_pos} and {end_pos} are same')
            return False

        if board.check_position(end_pos) != None and board.check_position(end_pos) != False and board.check_position(end_pos).color == self.color:    #end position shoud be of different color
            print(f'same color is present at {end_pos}')
            return False


        if not (delta_row == 0 and delta_col != 0) or not (delta_row != 0 and delta_col == 0):     #diagonal movement
            print('movement is diagonal, invalid move')
            return False

        if delta_row == 0:        # horizontal movement

            step = 1 if delta_col > 0 else -1          #+1 for right, -1 for left
            for col in range(start_col + step, end_col, step):
                if board.check_position((start_row, col)) != None and board.check_position((start_row, col)) != False:
                    print(f'there is some present between {start_col} and {end_col}')
                    return False              # some piece is present along the path

        elif delta_col == 0:        # Vertical movement

            step = 1 if delta_row > 0 else -1         #+1 for down, -1 for up
            for row in range(start_row + step, end_row, step):
                if board.check_position((row, start_col)) != None and board.check_position((row, start_col)) != False:
                    print(f'there is some piece present between {start_row} and {end_row}')
                    return False              # some piece is present along the path

        print('all conditions satisfied, valid move')
        return True          # if all checks pass (valid direction, no obstacles, valid destination), the move is valid


In [4]:
#bishops sub class of piece
class Bishop(Piece):
    def __init__(self, color, symbol):
        super().__init__(color, symbol)

    def is_valid_move(self, board, start_pos, end_pos):
        start_row, start_col = start_pos
        end_row, end_col = end_pos
        delta_row = end_row - start_row
        delta_col = end_col - start_col
        abs_delta_row = abs(delta_row)
        abs_delta_col = abs(delta_col)

        if board.check_position(end_pos) == False:      #end position should be on board
            print(f'{end_pos} is not on board')
            return False
        if start_pos == end_pos:      #end position should be different than the starting position
            print(f'{start_pos} and {end_pos} are same')
            return False

        if board.check_position(end_pos) != None and board.check_position(end_pos) != False and board.check_position(end_pos).color == self.color:    #end position shoud be of different color
            print(f'same color is present at {end_pos}')
            return False


        if abs(delta_row) != abs(delta_col) or delta_row == 0 or delta_col == 0:        #not diagonal move
            print('movement is not diagonal, invalid move')
            return False

        row_step = 1 if delta_row > 0 else -1         # +1 for down, -1 for up
        col_step = 1 if delta_col > 0 else -1         # +1 for right, -1 for left

        current_row = start_row + row_step
        current_col = start_col + col_step

        while current_row != end_row:
            if board.check_position((current_row, current_col)) != None and  board.check_position((current_row, current_col)) != False:
                print(f'there is some piece present between {start_col} and {end_col}')
                return False

            current_row += row_step
            current_col += col_step
        print('all conditions satisfied, valid move')
        return True


In [5]:
#knight sub class of piece
class Knight(Piece):
    def __init__(self, color, symbol):
        super().__init__(color, symbol)

    def is_valid_move(self, board, start_pos, end_pos):
        start_row, start_col = start_pos
        end_row, end_col = end_pos
        delta_row = end_row - start_row
        delta_col = end_col - start_col
        abs_delta_row = abs(delta_row)
        abs_delta_col = abs(delta_col)

        if board.check_position(end_pos) == False:      #end position should be on board
            print(f'{end_pos} is not on board')
            return False
        if start_pos == end_pos:      #end position should be different than the starting position
            print(f'{start_pos} and {end_pos} are same')
            return False

        if board.check_position(end_pos) != None and board.check_position(end_pos) != False and board.check_position(end_pos).color == self.color:    #end position shoud be of different color
            print(f'same color is present at {end_pos}')
            return False

        if (abs_delta_row == 2 and abs_delta_col == 1) or (abs_delta_row == 1 and abs_delta_col == 2):
            print('move is L shaped, valid move')
            return True
        else:
            print('move is not L shaped, invalid move')
            return False

In [6]:
#queen sub class of piece
class Queen(Piece):
    def __init__(self, color, symbol):
        super().__init__(color, symbol)

    def is_valid_move(self, board, start_pos, end_pos):
        start_row, start_col = start_pos
        end_row, end_col = end_pos
        delta_row = end_row - start_row
        delta_col = end_col - start_col
        abs_delta_row = abs(delta_row)
        abs_delta_col = abs(delta_col)

        if board.check_position(end_pos) == False:      #end position should be on board
            print(f'{end_pos} is not on board')
            return False
        if start_pos == end_pos:      #end position should be different than the starting position
            print(f'{start_pos} and {end_pos} are same')
            return False

        if board.check_position(end_pos) != None and board.check_position(end_pos) != False and board.check_position(end_pos).color == self.color:    #end position shoud be of different color
            print(f'same color is present at {end_pos}')
            return False


        is_hor_ver = (delta_row == 0 and delta_col != 0) or (delta_row != 0 and delta_col == 0)
        is_diagonal = abs(delta_row) == abs(delta_col) and delta_row != 0
        if not (is_hor_ver or is_diagonal):
            print('movement is not diagonal or horizontal or vertical, invalid move')
            return False

        if is_hor_ver:
            if delta_row == 0:
                step = 1 if delta_col > 0 else -1
                for col in range(start_col + step, end_col, step):
                    if board.check_position((start_row, col)) != None and board.check_position((start_row, col)) != False:
                        print(f'there is some piece present between {start_col} and {end_col}')
                        return False
            elif delta_col == 0:
                step = 1 if delta_row > 0 else -1
                for row in range(start_row + step, end_row, step):
                    if board.check_position((row, start_col)) != None and board.check_position((row, start_col)) != False:
                        print(f'there is some piece present between {start_row} and {end_row}')
                        return False

        elif is_diagonal:
            row_step = 1 if delta_row > 0 else -1
            col_step = 1 if delta_col > 0 else -1
            current_row = start_row + row_step
            current_col = start_col + col_step

            while current_row != end_row:
                if board.check_position((current_row, current_col)) != None and board.check_position((current_row, current_col)) != False:
                    print(f'there is some piece present between {start_row},{start_col} and {end_row},{end_col}')
                    return False

                current_row += row_step
                current_col += col_step
        print('all conditions satisfied, valid move')
        return True


In [7]:
#king sub class of piece
class King(Piece):
    def __init__(self, color, symbol):
        super().__init__(color, symbol)
        self.moved = False

    def is_valid_move(self, board, start_pos, end_pos):
        start_row, start_col = start_pos
        end_row, end_col = end_pos
        delta_row = end_row - start_row
        delta_col = end_col - start_col
        abs_delta_row = abs(delta_row)
        abs_delta_col = abs(delta_col)

        if board.check_position(end_pos) == False:      #end position should be on board
            print(f'{end_pos} is not on board')
            return False
        if start_pos == end_pos:      #end position should be different than the starting position
            print(f'{start_pos} and {end_pos} are same')
            return False

        if board.check_position(end_pos) != None and board.check_position(end_pos) != False and board.check_position(end_pos).color == self.color:    #end position shoud be of different color
            print(f'same color is present at {end_pos}')
            return False


        if abs_delta_row <= 1 and abs_delta_col <= 1:
            print('valid move')
            return True
        print('invalid move')
        return False

In [8]:
#Board Class

class Board():
  def __init__(self):
    self.grid= [[None for _ in range(8)] for _ in range(8)]

    self.place_piece(Rook('black', 'R'),(0,0))
    self.place_piece(Knight('black', 'N'),(0,1))
    self.place_piece(Bishop('black', 'B'),(0,2))
    self.place_piece(Queen('black', 'Q'),(0,3))
    self.place_piece(King('black', 'K'),(0,4))
    self.place_piece(Bishop('black', 'B'),(0,5))
    self.place_piece(Knight('black', 'N'),(0,6))
    self.place_piece(Rook('black', 'R'),(0,7))
    for i in range(8):
      self.place_piece(Pawn('black', 'P'),(1,i))

    self.place_piece(Rook('white', 'R'),(7,0))
    self.place_piece(Knight('white', 'N'),(7,1))
    self.place_piece(Bishop('white', 'B'),(7,2))
    self.place_piece(Queen('white', 'Q'),(7,3))
    self.place_piece(King('white', 'K'),(7,4))
    self.place_piece(Bishop('white', 'B'),(7,5))
    self.place_piece(Knight('white', 'N'),(7,6))
    self.place_piece(Rook('white', 'R'),(7,7))
    for i in range(8):
      self.place_piece(Pawn('white', 'P'),(6,i))


  def place_piece(self, piece, position):
    self.grid[position[0]][position[1]]=piece
    piece.position=position

  def remove_piece(self, piece):
    self.grid[piece.position[0]][piece.position[1]]=None
    piece.position=None

  def check_position(self, position):
    if position[0]<0 or position[0]>7 or position[1]<0 or position[1]>7:
      print(f'{position} is not on board')
      return False
    elif self.grid[position[0]][position[1]]==None:
      print(f'{position} is empty')
      return None
    else:
      print(f'{self.grid[position[0]][position[1]]} is at {position}')
      return self.grid[position[0]][position[1]]



  def move_piece(self, start_pos, end_pos):

        piece_to_move = self.check_position(start_pos)
        if piece_to_move is None:
            print(f"no piece at {start_pos}.")
            return False
        if piece_to_move is False:
            print(f"start position {start_pos} is not the board.")
            return False

        if piece_to_move.is_valid_move(self, start_pos, end_pos):
            self.remove_piece(piece_to_move)
            self.place_piece(piece_to_move, end_pos)

            if hasattr(piece_to_move, 'moved'):
                piece_to_move.moved = True

            print(f"successfully moved {piece_to_move.symbol} from {start_pos} to {end_pos}")
            return True
        else:
            print(f"Invalid move for {piece_to_move.symbol} from {start_pos} to {end_pos}")
            return False


In [9]:
#Player Class
class Player():
     def __init__(self, color):
         self.color = color

     def make_move(self, board, start_pos, end_pos):
         piece_to_move = board.check_position(start_pos)
         if piece_to_move is None:
             print(f"No piece at {start_pos}.")
             return False
         if piece_to_move is False:
             print(f"Start position {start_pos} is not on the board.")
             return False
         if piece_to_move.color != self.color:
             print(f"The piece at {start_pos} is not your color ({self.color}).")
             return False
         return board.move_piece(start_pos, end_pos)

In [10]:
class ChessGame():
    def __init__(self):
        self.board = Board()
        self.player1 = Player('white')
        self.player2 = Player('black')
        self.current_player = self.player1

    def display_board(self):
        print("\n  0 1 2 3 4 5 6 7")
        print(" +-----------------+")

        for r_idx in range(7, -1, -1):
            pieces_in_row_chars = []
            for c_idx in range(8):
                piece = self.board.grid[r_idx][c_idx]
                char = piece.symbol.lower() if piece and piece.color == 'black' else (piece.symbol if piece else '.')
                pieces_in_row_chars.append(char)
            row_display = f"{r_idx}| " + " ".join(pieces_in_row_chars) + " |"
            print(row_display)

        print(" +-----------------+")


    def _get_player_input(self, prompt):
        while True:
            try:
                user_input = input(prompt).strip()
                parts = user_input.split(',')

                if len(parts) != 2:
                    print("Invalid format. Please enter coordinates like '6,4' (row,col).")
                    continue

                row = int(parts[0])
                col = int(parts[1])

                if not (0 <= row <= 7 and 0 <= col <= 7):
                    print("Coordinates out of range. Row and column must be between 0 and 7.")
                    continue

                return (row, col)

            except ValueError:
                print("Invalid input. Please enter numbers for row and column (e.g., '6,4').")
            except IndexError:
                print("Invalid input. Please enter valid coordinates (e.g., '6,4').")


    def start(self):
        print("Welcome to Python Chess!")

        game_over = False
        while not game_over:
            self.display_board()
            print(f"\n{self.current_player.color.capitalize()}'s turn.")

            start_pos = self._get_player_input("Enter the starting position of the piece to move (e.g., 6,4): ")
            end_pos = self._get_player_input("Enter the ending position for the move (e.g., 4,5): ")

            print(f"Attempting to move from {start_pos} to {end_pos}")

            move_successful = self.current_player.make_move(self.board, start_pos, end_pos)

            if move_successful:
                print("Move successful!")
                self.current_player = self.player2 if self.current_player == self.player1 else self.player1
            else:
                print("Move invalid. Please try again.")


            if input("do you want to continue? (yes/no): ").lower() != 'y':
                game_over = True

In [11]:
    game = ChessGame()
    game.start()


Welcome to Python Chess!

  0 1 2 3 4 5 6 7
 +-----------------+
7| R N B Q K B N R |
6| P P P P P P P P |
5| . . . . . . . . |
4| . . . . . . . . |
3| . . . . . . . . |
2| . . . . . . . . |
1| p p p p p p p p |
0| r n b q k b n r |
 +-----------------+

White's turn.
Enter the starting position of the piece to move (e.g., 6,4): 6,3
Enter the ending position for the move (e.g., 4,5): 4,3
Attempting to move from (6, 3) to (4, 3)
<__main__.Pawn object at 0x7c2da1096610> is at (6, 3)
<__main__.Pawn object at 0x7c2da1096610> is at (6, 3)
(4, 3) is empty
(4, 3) is empty
(4, 3) is empty
(5, 3) is empty
(4, 3) is empty and there is nothing in between, so 2 steps can be successfully taken
successfully moved P from (6, 3) to (4, 3)
Move successful!
do you want to continue? (yes/no): y

  0 1 2 3 4 5 6 7
 +-----------------+
7| R N B Q K B N R |
6| P P P . P P P P |
5| . . . . . . . . |
4| . . . P . . . . |
3| . . . . . . . . |
2| . . . . . . . . |
1| p p p p p p p p |
0| r n b q k b n r |
 +---

#Ride-Sharing Simulation
Create a simulation of rides:

Driver and Rider classes.

Ride class with pickup, drop, and fare calculation.

Logic for matching riders to drivers (e.g., based on availability).

In [ ]:
#Driver class
class Driver():
  def __init__(self, name, vehicle, location, availability):
    self.name = name
    self.vehicle = vehicle
    self.location = location
    self.availability = availability
  def update_location(self, new_location):
    self.location = new_location
  def set_availability(self, status):
    if status == 'available' or status == 'unavailable':
      self.availability = status
      return True
    else:
      print('invalid status')
      return False

In [ ]:
#Rider class
class Rider():
  def __init__(self, name, location):
    self.name = name
    self.location = location

  def request_ride(self, drop_location):
        print(f"{self.name} at {self.location} is requesting a ride to {drop_location}.")
        return {
            "rider": self.name,
            "pickup": self.location,
            "drop_off": drop_location
        }


In [ ]:
#Ride class
class Ride():
  def __init__(self, driver, rider, pickup, drop, fare):
    self.driver = driver
    self.rider = rider
    self.pickup = pickup
    self.drop = drop
    self.fare = None
    self.status = 'Pending'
  def total_distance(self):
    return abs(self.pickup[0] - self.drop[0]) + abs(self.pickup[1] - self.drop[1])
  def calculate_fare(self):
    self.fare=self.total_distance()*10
    return self.fare


In [ ]:
#Ride system class
class RideSystem:
    def __init__(self, drivers=None, riders=None, active_rides=None, completed_rides=None):
        self.drivers = list(drivers) if drivers is not None else []
        self.riders = list(riders) if riders is not None else []
        self.active_rides = list(active_rides) if active_rides is not None else []
        self.completed_rides = list(completed_rides) if completed_rides is not None else []
    def register_driver(self, driver):
        self.drivers.append(driver)

    def register_rider(self, rider):
        self.riders.append(rider)

    @staticmethod
    def calculate_distance(pickup, drop):
      return abs(pickup[0] - drop[0]) + abs(pickup[1] - drop[1])

    def request_ride(self, rider, drop_off_location):
        print(f"{rider.name} requests a ride from {rider.location} to {drop_off_location}.")
        nearest_driver = None
        min_distance = float('inf')
        for driver in self.drivers:
            if driver.availability == 'available':
                distance = self.calculate_distance(rider.location, driver.location)
                if distance < min_distance:
                    min_distance = distance
                    nearest_driver = driver
        if nearest_driver:
            r = Ride(nearest_driver, rider, rider.location, drop_off_location, None)
            r.calculate_fare()
            nearest_driver.set_availability('unavailable')
            self.active_rides.append(r)
            print(f"Ride accepted! {nearest_driver.name} is assigned to {rider.name}. \n Fare: ${r.fare}")
            return r
        else:
            print("No drivers available for this request.")
            return None

In [ ]:
#random drivers
d1 = Driver("Ali", "Bike", (0, 0),'available')
d2 = Driver("Bilal", "Car", (10, 1),'available')
d3 = Driver("Ahmad", "Bike", (5, 13),'available')
d4 = Driver("Haseeb", "Bike", (2, 20),'unavailable')
d5 = Driver("Umer", "Bike", (0, 10),'available')

In [ ]:
#create riders
r1=Rider('Danyal', (2,5))
r2=Rider('Zunair',(3,15))
r3=Rider('Asim',(1,20))

In [ ]:
R_S=RideSystem()

In [ ]:
#register
R_S.register_driver(d1)
R_S.register_driver(d2)
R_S.register_driver(d3)
R_S.register_driver(d4)
R_S.register_driver(d5)

R_S.register_rider(r1)
R_S.register_rider(r2)
R_S.register_rider(r3)

In [ ]:
#to check distance of rider from each driver in system
def driver_distances_for_rider(r_s, t_rider):
    print(f"Distance Analysis for Rider: {t_rider.name}")
    print(f"Rider's pickup location: {t_rider.location}")

    for driver in r_s.drivers:
        distance = r_s.calculate_distance(t_rider.location, driver.location)
        availability_status = "Available" if driver.availability == 'available' else "Unavailable"
        print(f"- Driver {driver.name} at {driver.location}: Distance = {distance}")
for r in R_S.riders:
  driver_distances_for_rider(R_S, r)
  print('<----->')

Distance Analysis for Rider: Danyal
Rider's pickup location: (2, 5)
- Driver Ali at (0, 0): Distance = 7
- Driver Bilal at (10, 1): Distance = 12
- Driver Ahmad at (5, 13): Distance = 11
- Driver Haseeb at (2, 20): Distance = 15
- Driver Umer at (0, 10): Distance = 7
<----->
Distance Analysis for Rider: Zunair
Rider's pickup location: (3, 15)
- Driver Ali at (0, 0): Distance = 18
- Driver Bilal at (10, 1): Distance = 21
- Driver Ahmad at (5, 13): Distance = 4
- Driver Haseeb at (2, 20): Distance = 6
- Driver Umer at (0, 10): Distance = 8
<----->
Distance Analysis for Rider: Asim
Rider's pickup location: (1, 20)
- Driver Ali at (0, 0): Distance = 21
- Driver Bilal at (10, 1): Distance = 28
- Driver Ahmad at (5, 13): Distance = 11
- Driver Haseeb at (2, 20): Distance = 1
- Driver Umer at (0, 10): Distance = 11
<----->


In [ ]:
R1=R_S.request_ride(r1, (5,5))

print(f'Total distance is {R1.total_distance()}')

Danyal requests a ride from (2, 5) to (5, 5).
Ride accepted! Ali is assigned to Danyal. 
 Fare: $30
Total distance is 3


In [ ]:
R2=R_S.request_ride(r2, (10,26))

print(f'Total distance is {R2.total_distance()}')

Zunair requests a ride from (3, 15) to (10, 26).
Ride accepted! Ahmad is assigned to Zunair. 
 Fare: $180
Total distance is 18


In [ ]:
R3=R_S.request_ride(r3, (1,1))

print(f'Total distance is {R3.total_distance()}')

Asim requests a ride from (1, 20) to (1, 1).
Ride accepted! Umer is assigned to Asim. 
 Fare: $190
Total distance is 19


In [ ]:
print(f"Total active rides: {len(my_ride_system.active_rides)}")
print(f"Driver Ali final availability: {d1.availability}")
print(f"Driver Bilal final availability: {d2.availability}")
print(f"Driver Ahmad final availability: {d3.availability}")
print(f"Driver Haseeb final availability: {d4.availability}")
print(f"Driver Umer final availability: {d5.availability}")

Total active rides: 3
Driver Ali final availability: unavailable
Driver Bilal final availability: available
Driver Ahmad final availability: unavailable
Driver Haseeb final availability: unavailable
Driver Umer final availability: unavailable
